# 2026-11-20 EDA on FOODS: what the data looks like per store and state
Messy exploring is allowed here. Anything worth keeping moves into `src/m5food`.

Question for this notebook: *how do FOODS sales look across the 10 stores and 3 states, and which direction is worth digging into?*

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from m5food import data, plots
plots.set_style()

## Load
Raw files must be in `data/raw/` (see README).

In [ ]:
sales = data.load_sales_foods()
cal = data.load_calendar()
prices = data.load_prices()
print(sales.shape)
print(cal.shape)
print(prices.shape)

## 0. Sanity check
Expected: 1,437 items x 10 stores = 14,370 rows, 1,941 day columns.

In [ ]:
days = data.day_columns(sales)
print(sales.shape, len(days), 'days')
sales.groupby(['state_id', 'store_id']).size().rename('series').to_frame().T

Helper: the wide table summed per group, with real dates on the index. Every chart below uses this.

In [ ]:
dates = cal.set_index('d').loc[days, 'date'].values   # d_1 ... d_1941 -> dates

def daily_by(col):
    """Total units per day for each value of `col`. Rows = dates, columns = groups."""
    out = sales.groupby(col)[days].sum().T
    out.index = pd.DatetimeIndex(dates, name='date')
    return out

by_state = daily_by('state_id')
by_store = daily_by('store_id')
STATE_OF = sales.drop_duplicates('store_id').set_index('store_id')['state_id']
STATE_COLOR = dict(zip(['CA', 'TX', 'WI'], plots.PALETTE))
by_store

## 1. How much does each store sell in total?
Look for: is one state or store much bigger than the rest?

In [ ]:
import plotly.graph_objects as go
from matplotlib.colors import to_rgb, to_hex

# Plain-English names
STATE_NAME = {'CA': 'California', 'TX': 'Texas', 'WI': 'Wisconsin'}
def store_name(s): st, n = s.split('_'); return f'{STATE_NAME[st]} store {n}'
def dept_name(d):  return f'Food dept {d.split("_")[1]}'
def m(x): return f'{x / 1e6:.1f}M'

# Colours: state base -> a different shade per store -> a different tint per department
STATE_BASE = {'CA': plots.PALETTE[0], 'TX': plots.PALETTE[1], 'WI': plots.PALETTE[2]}
def shade(c, k):
    """k < 0 darkens towards black, k > 0 lightens towards white."""
    target = np.zeros(3) if k < 0 else np.ones(3)
    return to_hex(np.array(to_rgb(c)) * (1 - abs(k)) + target * abs(k))
STORE_SHIFT = [-0.35, -0.12, 0.12, 0.3]                             # store 1..4 within a state
DEPT_TINT = {'FOODS_1': 0.6, 'FOODS_2': 0.35, 'FOODS_3': 0.1}         # dept 1 lightest, dept 3 darkest
store_color = {s: shade(STATE_BASE[s.split('_')[0]], STORE_SHIFT[int(s.split('_')[1]) - 1])
               for s in sales.store_id.unique()}

t = sales.groupby(['state_id', 'store_id', 'dept_id'])[days].sum().sum(axis=1).rename('units').reset_index()
items = sales.groupby('dept_id').item_id.nunique()
total = t.units.sum()
st_tot = t.groupby('state_id').units.sum()
sto_tot = t.groupby('store_id').units.sum()
n_dept = t.groupby('store_id').size()
n_st = t.groupby('state_id').size()

# size = box area (equal per department, NOT units); text = short, shown in box; hover = full sentence
nodes = [dict(id='ALL', parent='', label='All FOODS', size=len(t), color='#e9ecef',
              text=f'{m(total)} units sold, 2011-2016', hover=f'{m(total)} FOODS units sold, 2011-2016')]
for st, u in st_tot.items():
    nodes.append(dict(id=st, parent='ALL', label=STATE_NAME[st], size=n_st[st], color=STATE_BASE[st],
                      text=f'{u / total:.0%} of all FOODS units ({m(u)})',
                      hover=f'{STATE_NAME[st]} sold {u / total:.0%} of all FOODS units ({m(u)})'))
for sto, u in sto_tot.items():
    st = sto.split('_')[0]
    nodes.append(dict(id=sto, parent=st, label=store_name(sto), size=n_dept[sto], color=store_color[sto],
                      text=f'{u / st_tot[st]:.0%} of {STATE_NAME[st]} ({m(u)})',
                      hover=f'{store_name(sto)} sold {u / st_tot[st]:.0%} of {STATE_NAME[st]}\'s FOODS units ({m(u)})'))
for r in t.itertuples():
    share = r.units / sto_tot[r.store_id]
    nodes.append(dict(id=f'{r.store_id}/{r.dept_id}', parent=r.store_id, label=dept_name(r.dept_id),
                      size=1, color=shade(store_color[r.store_id], DEPT_TINT[r.dept_id]),
                      text=f'{share:.0%} of this store<br>{items[r.dept_id]} items<br>{m(r.units)} units',
                      hover=f'{share:.0%} of {store_name(r.store_id)}\'s FOODS units came from '
                            f'{dept_name(r.dept_id)}<br>({items[r.dept_id]} items, {m(r.units)} units)'))
nodes = pd.DataFrame(nodes)

fig = go.Figure(go.Treemap(
    ids=nodes.id, parents=nodes.parent, labels=nodes.label, values=nodes['size'],
    text=nodes.text, customdata=nodes.hover, branchvalues='total',
    marker=dict(colors=nodes.color, line=dict(color='white', width=2)),
    texttemplate='<b>%{label}</b><br>%{text}',
    hovertemplate='<b>%{label}</b><br>%{customdata}<extra></extra>'))

how_to_read = ('<span style="font-size:13px;color:#444">'
               '<b>How to read:</b> boxes are equal size on purpose; the <b>numbers</b> show how much each sold. '
               'Each store has its own shade; inside it, Food dept 1 is lightest and Food dept 3 darkest.<br>'
               '<b>Food dept 1 / 2 / 3</b> are anonymised in the data (we do not know what they sell); every store '
               f'lists the same {items["FOODS_1"]} / {items["FOODS_2"]} / {items["FOODS_3"]} items. '
               'Click a box to zoom in; click the bar at the top to go back.</span>')

fig.update_layout(
    title=dict(text='<b>Where FOODS units were sold, 2011-2016</b><br>' + how_to_read,
               x=0.01, xanchor='left', y=0.98, yref='container', yanchor='top'),
    width=1300, height=800, margin=dict(t=120, l=10, r=10, b=10), template='simple_white')
fig.show()


In [ ]:
import re
import plotly.graph_objects as go

# ---- 1. Drop the empty "All FOODS" column: its total moves into the title ----
root_id = nodes.loc[nodes.parent == '', 'id'].iloc[0]
total_units = nodes.loc[nodes.id == root_id, 'size'].iloc[0]
n = nodes[nodes.id != root_id].copy()
n['parent'] = n['parent'].where(n['parent'] != root_id, '')          # states become the first column

# ---- 2. Work out each box's column (state / store / department) and its state ----
parent_of = dict(zip(n.id, n.parent))
def depth(i):  return 0 if parent_of[i] == '' else 1 + depth(parent_of[i])
def state(i):  return i if parent_of[i] == '' else state(parent_of[i])
n['depth'] = n.id.map(depth)
n['state'] = n.id.map(state)

# ---- 3. Colour rule: colour = state, lighter shade = further right; same department = same tint everywhere ----
BY_NAME = {'California': '#2a6f97', 'Texas': '#e07a1f', 'Wisconsin': '#4c9f70'}               # same colours as the rest of the notebook
STATE_COLOUR = {r.id: BY_NAME[r.label] for r in n[n.depth == 0].itertuples()}
def mix(hex_, k):                                                    # k = share of white mixed in
    r, g, b = (int(hex_[i:i+2], 16) for i in (1, 3, 5))
    return '#%02x%02x%02x' % tuple(round(c + (255 - c) * k) for c in (r, g, b))
def text_on(hex_):                                                   # dark text on light boxes, white on dark
    r, g, b = (int(hex_[i:i+2], 16) for i in (1, 3, 5))
    return '#1f1f1f' if 0.299*r + 0.587*g + 0.114*b > 150 else 'white'
DEPT_TINT = {1: 0.80, 2: 0.65, 3: 0.50}                              # dept 1 lightest, dept 3 strongest
def colour(row):
    base = STATE_COLOUR[row.state]
    if row.depth == 0: return base
    if row.depth == 1: return mix(base, 0.30)
    d = int(re.search(r'(\d)\s*$', row.label).group(1))             # "Food dept 3" -> 3
    return mix(base, DEPT_TINT[d])
n['colour'] = n.apply(colour, axis=1)

# ---- 4. One short label per box, sized to fit its row ----
def short(row):
    sub = re.sub(r'[\d,]+ items · ', '', row.line)                  # items count stays in the hover
    if row.depth == 2: return f'<b>{row.label}</b>  ·  {sub}'      # one line: thin rows stay readable
    return f'<b>{row.label}</b><br>{sub}'
n['line'] = n.text.str.replace('<br>', ' · ')
n['label_text'] = n.apply(short, axis=1)

fig = go.Figure(go.Icicle(
    ids=n.id, parents=n.parent, labels=n.label, values=n['size'],
    text=n.label_text, customdata=n.hover, branchvalues='total',
    tiling=dict(orientation='h'),
    marker=dict(colors=n.colour, line=dict(color='white', width=2)),
    texttemplate='%{text}', textposition='middle left',
    textfont=dict(size=15, color=[text_on(c) for c in n.colour]),
    insidetextfont=dict(size=15, color=[text_on(c) for c in n.colour]),
    hovertemplate='<b>%{label}</b><br>%{customdata}<extra></extra>',
    pathbar=dict(visible=False)))

# ---- 5. Column headers, so the reader never has to guess what a column is ----
for x, name in [(1/6, 'STATE'), (3/6, 'STORE'), (5/6, 'DEPARTMENT')]:
    fig.add_annotation(x=x, y=1.0, xref='paper', yref='paper', yanchor='bottom', showarrow=False,
                       text=f'<b>{name}</b>', font=dict(size=14, color='#6c757d'))

fig.update_layout(
    title=dict(
        text=f'<b>Department 3 sells about 7 of every 10 units in every store</b><br>'
             f'<span style="font-size:15px;color:#555">FOODS, {total_units/1e6:.1f}M units sold, 2011-2016. '
             f'Box height = units sold. Click a box to zoom in.</span>',
        x=0.01, xanchor='left', y=0.97, yanchor='top'),
    width=1250, height=900, margin=dict(t=130, l=10, r=10, b=10),
    template='simple_white', uniformtext=dict(minsize=11, mode='show'))
fig.show()


In [ ]:
tot = by_store.sum().sort_values(ascending=False) / 1e6
fig, ax = plt.subplots()
ax.bar(tot.index, tot.values, color=[STATE_COLOR[STATE_OF[s]] for s in tot.index])
plots.label_bars(ax, '{:.1f}')
ax.set_ylabel('units sold, millions (2011-2016)')
ax.set_title('Total FOODS units per store, coloured by state')
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in STATE_COLOR.values()]
ax.legend(handles, STATE_COLOR.keys(), frameon=False);

## 2. Daily sales per state over 5.4 years
Thin line = daily, thick = 28-day average. Look for: trend, yearly pattern, sudden drops to zero.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for st in by_state:
    ax.plot(by_state.index, by_state[st], lw=0.4, alpha=0.35, color=STATE_COLOR[st])
    ax.plot(by_state.index, by_state[st].rolling(28).mean(), lw=2.2, color=STATE_COLOR[st], label=st)
ax.set_ylabel('units per day'); ax.legend(frameon=False)
ax.set_title('FOODS units per day, by state');

## 3. Same, per store (grouped by state)
Look for: stores that do not follow their state; level shifts (a store suddenly growing or shrinking).

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

roll = by_store.rolling(28).mean()
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=['CA', 'TX', 'WI'])
for r, st in enumerate(['CA', 'TX', 'WI'], start=1):
    for i, store in enumerate([s for s in by_store if STATE_OF[s] == st]):
        fig.add_trace(go.Scatter(
            x=roll.index, y=roll[store], name=store, mode='lines',
            line=dict(color=plots.PALETTE[i], width=2), legendgroup=st, legendgrouptitle_text=st,
            customdata=by_store[store],
            hovertemplate=store + ': %{y:,.0f}/day (28d avg) · that day %{customdata:,}<extra></extra>'),
            row=r, col=1)
    fig.update_yaxes(title_text='units/day (28d avg)', row=r, col=1)

fig.update_layout(title='<b>FOODS units per store, 28-day average</b>', template='simple_white',
                  height=950, hovermode='x unified', legend=dict(groupclick='toggleitem'))
fig.show()


## 4. Weekly pattern
Average units per weekday, per state. Look for: how strong is the weekend effect, is it the same in all states?

In [ ]:
wd_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday = cal.set_index('d').loc[days, 'weekday'].values
wk = by_state.groupby(weekday).mean().loc[wd_order]
ax = wk.plot(kind='bar', color=[STATE_COLOR[s] for s in wk.columns], rot=0)
ax.set_ylabel('avg units per day'); ax.set_xlabel('')
ax.set_title('FOODS: average units by weekday and state'); ax.legend(frameon=False);

## 6. Departments: what is FOODS made of in each store?
FOODS_1, FOODS_2, FOODS_3. Look for: does one department dominate, does the mix change by store?

In [ ]:
import plotly.express as px

g = sales.groupby(['store_id', 'dept_id'])
mix = pd.DataFrame({'units': g[days].sum().sum(axis=1), 'items': g.size()}).reset_index()
mix['share'] = mix['units'] / mix.groupby('store_id')['units'].transform('sum') * 100

fig = px.bar(mix, x='store_id', y='share', color='dept_id',
             text=mix['share'].round(0).astype(int).astype(str) + '%',
             custom_data=['dept_id', 'items', 'units'],
             color_discrete_sequence=plots.PALETTE,
             labels={'store_id': '', 'share': '% of store units', 'dept_id': 'department'},
             title='FOODS department mix per store')
fig.update_traces(textposition='inside',
                  hovertemplate='<b>%{x} · %{customdata[0]}</b><br>'
                                'items: %{customdata[1]:,}<br>'
                                'units sold: %{customdata[2]:,.0f}<br>'
                                'share of store units: %{y:.1f}%<extra></extra>')
fig.update_layout(template='simple_white', height=520, legend_title_text='')
fig.show()


In [ ]:
MIN_SPREAD_DAYS = 7          # keep items whose earliest and latest store first-sale are at least this far apart

spread = (first_sale.max(axis=1) - first_sale.min(axis=1)).dt.days
far = first_sale[spread >= MIN_SPREAD_DAYS]
print(f'kept {len(far):,} of {len(first_sale):,} items (spread >= {MIN_SPREAD_DAYS} days)')
pd.Series({t: (spread >= t).sum() for t in [7, 28, 90, 180, 365]}, name='items kept').rename_axis('min spread (days)')
import plotly.express as px

order = far.min(axis=1).sort_values().index
days_in = (far.loc[order] - pd.Timestamp(dates[0])).apply(lambda c: c.dt.days)

fig = px.line(started, labels={'index': '', 'value': 'items that have sold at least once', 'variable': 'store'},
              title=f'When the {len(far):,} "far apart" items started selling, per store')
fig.add_hline(y=len(far), line_dash='dot', annotation_text=f'all {len(far):,} items')
fig.update_layout(template='simple_white', height=500)
fig.show()
import plotly.express as px

for dept in ['FOODS_1', 'FOODS_2', 'FOODS_3']:
    d = far[far.index.str.startswith(dept + '_')]
    order = d.min(axis=1).sort_values().index
    days_in = (d.loc[order] - pd.Timestamp(dates[0])).apply(lambda c: c.dt.days)

    fig = px.imshow(days_in, aspect='auto', color_continuous_scale='Blues', zmin=0, zmax=len(dates),
                    labels=dict(x='store', y='item', color='days after 29 Jan 2011'),
                    title=f'{dept}: first sale per store, {len(d):,} items with stores ≥{MIN_SPREAD_DAYS} days apart')
    fig.update_traces(customdata=d.loc[order].astype(str).to_numpy(),
                      hovertemplate='%{y} · %{x}<br>first sale: %{customdata}<br>day %{z}<extra></extra>')
    fig.update_yaxes(showticklabels=len(d) <= 60)      # show item names only when there are few enough to read
    fig.update_layout(template='simple_white', height=max(350, min(900, 6 * len(d))))
    fig.show()



## 7. How often does an item sell nothing?
Share of days with zero units, per item-store series. Counted only from each series' first sale (before that the item probably was not on the shelf). Look for: how many series are mostly zeros.

In [ ]:
X = sales[days].to_numpy()
first = (X > 0).argmax(axis=1)                       # first day with a sale
live = np.arange(X.shape[1]) >= first[:, None]       # days after the item appeared
zero_share = ((X == 0) & live).sum(1) / live.sum(1) * 100
sales['zero_share'] = zero_share

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].hist(zero_share, bins=50)
axes[0].set_xlabel('% of days with zero sales'); axes[0].set_ylabel('number of series')
axes[0].set_title('Zero days per series (all 14,370)')
med = sales.groupby('store_id')['zero_share'].median()
axes[1].bar(med.index, med.values, color=[STATE_COLOR[STATE_OF[s]] for s in med.index])
plots.label_bars(axes[1], '{:.0f}'); axes[1].set_ylabel('median % zero days')
axes[1].set_title('Median zero-day share per store')
axes[1].tick_params(axis='x', rotation=45)
fig.tight_layout();

In [ ]:
# 7b. Zero-day % (counted only from each series' first sale): what share of series falls in each band?
# Needs sales['zero_share'] from the cell above.
BANDS = [0, 20, 40, 60, 80, 100]
BAND_LABELS = ['0-20%', '20-40%', '40-60%', '60-80%', '80-100%']
BAND_COLORS = ['#d8e6f0', '#9ec2dc', '#5f97c0', '#2a6f97', '#173f5a']   # light = sells most days, dark = mostly zeros
sales['zero_band'] = pd.cut(sales['zero_share'], BANDS, labels=BAND_LABELS, include_lowest=True)

def band_share(by):
    """% of series in each zero-day band, one row per group."""
    t = pd.crosstab([sales[c] for c in by], sales['zero_band'], normalize='index') * 100
    t.index = [' / '.join(i) if isinstance(i, tuple) else i for i in t.index]
    return t[BAND_LABELS]

def stacked(ax, t, title):
    left = np.zeros(len(t))
    for lab, col in zip(BAND_LABELS, BAND_COLORS):
        ax.barh(t.index, t[lab], left=left, color=col, label=lab, edgecolor='white')
        for y, (l, w) in enumerate(zip(left, t[lab])):
            if w >= 5:                                           # label only segments wide enough to read
                ax.text(l + w / 2, y, f'{w:.0f}%', ha='center', va='center', fontsize=11,
                        color='black' if col in BAND_COLORS[:2] else 'white')
        left += t[lab].to_numpy()
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.grid(False)
    ax.set_title(title, fontsize=15); ax.set_xticks([])

tables = [(band_share(['dept_id']), 'By department'),
          (band_share(['state_id']), 'By state'),
          (band_share(['dept_id', 'state_id']), 'By department within state')]
fig, axes = plt.subplots(3, 1, figsize=(13, 11), gridspec_kw={'height_ratios': [3, 3, 9]})
for ax, (t, title) in zip(axes, tables):
    stacked(ax, t, title)
axes[0].legend(title='days with zero sales', ncol=5, frameon=False, loc='lower left',
               bbox_to_anchor=(0, 1.25), fontsize=11)
fig.suptitle('Share of item-store series by % of zero-sales days (from first sale)',
             fontsize=18, fontweight='bold', x=0.01, ha='left')
fig.tight_layout()

# Same numbers as a table, plus the median per group
summary = tables[2][0].round(0)
summary['median zero %'] = sales.groupby(['dept_id', 'state_id'])['zero_share'].median().round(0).to_numpy()
summary


In [ ]:
# 7c. Distribution of zero-day % per series (from first sale). Is it normal?
from statistics import NormalDist
z = sales['zero_share'].to_numpy()
mu, med, sd = z.mean(), np.median(z), z.std()
skew = ((z - mu) ** 3).mean() / sd ** 3

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [3, 2]})
ax = axes[0]
ax.hist(z, bins=50, range=(0, 100), density=True, alpha=0.75, label='series')
xs = np.linspace(0, 100, 300)
ax.plot(xs, [NormalDist(mu, sd).pdf(x) for x in xs], color='black', lw=2, ls='--',
        label='normal curve, same mean & SD')
ax.axvline(mu, color=plots.PALETTE[1], lw=2.5, label=f'mean {mu:.1f}%')
ax.axvline(med, color=plots.PALETTE[2], lw=2.5, label=f'median {med:.1f}%')
ax.set_xlabel('% of days with zero sales (from first sale)'); ax.set_ylabel('density')
ax.set_title('Zero-day share across 14,370 series'); ax.legend(frameon=False, fontsize=11)

# Q-Q plot: if the data were normal, the dots would sit on the straight line
ax = axes[1]
q = np.linspace(0.005, 0.995, 199)
ax.scatter([NormalDist(mu, sd).inv_cdf(p) for p in q], np.quantile(z, q), s=14)
ax.plot([0, 100], [0, 100], color='black', lw=1.5, ls='--')
ax.set_xlim(-10, 110); ax.set_ylim(-10, 110)
ax.set_xlabel('expected if normal (%)'); ax.set_ylabel('actual (%)')
ax.set_title('Q-Q plot vs normal')
fig.tight_layout()

pd.Series({'series': len(z), 'mean %': mu, 'median %': med, 'std %': sd, 'skew': skew,
           'p10 %': np.percentile(z, 10), 'p25 %': np.percentile(z, 25),
           'p75 %': np.percentile(z, 75), 'p90 %': np.percentile(z, 90)}).round(2)


In [ ]:
# 7d. Are zeros scattered (single days here and there) or bunched in long gaps?
# A "run" = consecutive zero days. Counted only after the first sale.
def zero_runs(row, start):
    """Lengths of every run of consecutive zeros from day `start` onwards."""
    is0 = np.r_[False, row[start:] == 0, False].astype(int)
    edges = np.diff(is0)
    return np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)

runs = [zero_runs(X[i], first[i]) for i in range(len(X))]
all_runs = np.concatenate(runs)
sales['longest_zero_run'] = [r.max() if len(r) else 0 for r in runs]
sales['ends_in_zeros'] = X[:, -1] == 0          # last day is zero: possibly still in a gap or delisted

RUN_BINS = [0, 1, 3, 7, 28, 90, np.inf]
RUN_LABELS = ['1 day', '2-3 days', '4-7 days', '8-28 days', '29-90 days', '90+ days']
run_band = pd.cut(all_runs, RUN_BINS, labels=RUN_LABELS)
# Share of zero DAYS (not runs) that sit in each run length: a 60-day gap counts 60 times
days_in = pd.Series(all_runs).groupby(run_band, observed=False).sum() / all_runs.sum() * 100
runs_in = pd.Series(run_band).value_counts(normalize=True).reindex(RUN_LABELS) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ax = axes[0]
x = np.arange(len(RUN_LABELS))
ax.bar(x - 0.2, runs_in.values, 0.4, label='% of runs')
ax.bar(x + 0.2, days_in.values, 0.4, label='% of zero days')
for xi, a, b in zip(x, runs_in.values, days_in.values):
    ax.text(xi - 0.2, a + 1, f'{a:.0f}', ha='center', fontsize=11)
    ax.text(xi + 0.2, b + 1, f'{b:.0f}', ha='center', fontsize=11)
ax.set_xticks(x, RUN_LABELS, rotation=20); ax.set_ylabel('%')
ax.set_title('How long are the zero runs?'); ax.legend(frameon=False)

ax = axes[1]
GAP_BINS = [-1, 7, 14, 28, 60, 90, 180, np.inf]
GAP_LABELS = ['≤7', '8-14', '15-28', '29-60', '61-90', '91-180', '180+']
gap = pd.cut(sales['longest_zero_run'], GAP_BINS, labels=GAP_LABELS).value_counts(normalize=True).reindex(GAP_LABELS) * 100
ax.bar(GAP_LABELS, gap.values)
plots.label_bars(ax, '{:.0f}%')
ax.set_xlabel('longest zero run in the series (days)'); ax.set_ylabel('% of series')
ax.set_title('Longest gap per series')
fig.tight_layout()

pd.Series({
    'zero runs in total': len(all_runs),
    'median run length (days)': np.median(all_runs),
    'series with a gap > 28 days (%)': (sales['longest_zero_run'] > 28).mean() * 100,
    'series with a gap > 90 days (%)': (sales['longest_zero_run'] > 90).mean() * 100,
    'series whose last day is zero (%)': sales['ends_in_zeros'].mean() * 100,
}).round(1)


In [ ]:
import plotly.express as px

lr = sales['longest_zero_run']
fig = px.histogram(lr, x='longest_zero_run', log_y=True,
                   labels={'longest_zero_run': 'longest run of zero days in the series (days)'},
                   title='Longest zero gap per series')
fig.update_traces(xbins=dict(start=0, size=7))   # one bar = one week
fig.update_layout(yaxis_title='number of series (log scale)', showlegend=False,
                  bargap=0.05, template='simple_white', height=500)
fig.show()
